# P1 - Notebook de Predicciones (Competición)
Cristian Dulugeac 100522196

Este notebook carga el modelo final y genera predicciones sobre los datos de competición.

---

In [1]:
import pandas as pd
import numpy as np
import joblib
from sklearn.preprocessing import LabelEncoder

NIA  = 100522196
last_two = NIA % 100
xx = (last_two // 10) + (last_two % 10)
print(f'Fichero de competición: bank_competition.pkl')

Fichero de competición: bank_competition.pkl


In [2]:
# Cargar modelo final
model = joblib.load('modelo_final.joblib')
print('Modelo cargado:', model)

# Cargar datos de competición
df_comp = pd.read_pickle(f'bank_competition.pkl')
print('Shape datos competición:', df_comp.shape)
df_comp.head()

Modelo cargado: Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('cat',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('ohe',
                                                                   OneHotEncoder(handle_unknown='ignore'))]),
                                                  ['job', 'marital',
                                                   'education', 'default',
                                                   'housing', 'loan', 'contact',
                                                   'month', 'poutcome']),
                                                 ('num',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer()),
    

,age,job,marital,education,default,balance,housing,loan,contact,day,month,duration,campaign,pdays,previous,poutcome
5553,43,management,married,tertiary,no,78,yes,no,cellular,21,nov,36,1,109,1,other
915,34,housemaid,married,secondary,no,0,yes,no,unknown,30,oct,154,1,-1,0,unknown
7652,54,technician,married,secondary,no,3323,yes,yes,cellular,8,apr,59,3,-1,0,unknown
5065,43,blue-collar,single,primary,no,-399,no,yes,cellular,28,jul,662,3,-1,0,unknown
3338,35,blue-collar,married,secondary,no,262,no,no,cellular,15,mar,427,1,181,3,success


In [3]:
# Preprocesamiento de pdays (igual que en el notebook principal)
df_comp['contacted_before'] = (df_comp['pdays'] != -1).astype(int)
df_comp['pdays_clean']      = df_comp['pdays'].clip(lower=0)
df_comp = df_comp.drop(columns=['pdays'])
print('Shape tras preproceso:', df_comp.shape)

Shape tras preproceso: (162, 17)


In [4]:
# Predicciones
le = LabelEncoder()
le.fit(['no', 'yes'])  # mismo orden que en el notebook principal

y_pred     = model.predict(df_comp)
y_labels   = le.inverse_transform(y_pred)

predicciones = pd.DataFrame({'deposit': y_labels})
predicciones.to_csv('predicciones.csv', index=False)

print('predicciones.csv generado.')
print(predicciones['deposit'].value_counts())
predicciones.head(10)

predicciones.csv generado.
deposit
no     91
yes    71
Name: count, dtype: int64


,deposit
0,no
1,no
2,no
3,yes
4,yes
5,no
6,yes
7,no
8,yes
9,no


In [5]:
# Verificación: mismas predicciones que el pipeline en 2 instancias de ejemplo
ejemplo = df_comp.iloc[:2].copy()
pred_ejemplo = model.predict(ejemplo)
print('Predicciones ejemplo (pipeline):', le.inverse_transform(pred_ejemplo))
print('Predicciones ejemplo (csv):     ', list(predicciones['deposit'].iloc[:2]))

Predicciones ejemplo (pipeline): ['no' 'no']
Predicciones ejemplo (csv):      ['no', 'no']


In [6]:
modelo = joblib.load('modelo_final.joblib')
le = LabelEncoder().fit(['no', 'yes'])

# Dos clientes de ejemplo para verificar que el modelo da las mismas predicciones que el pipeline
cliente1 = pd.DataFrame([{
    'age': 35, 'job': 'technician', 'marital': 'married',
    'education': 'secondary', 'default': 'no', 'balance': 1500,
    'housing': 'yes', 'loan': 'no', 'contact': 'cellular',
    'day': 10, 'month': 'may', 'duration': 300, 'campaign': 1,
    'pdays': -1, 'previous': 0, 'poutcome': 'unknown'
}])
cliente2 = pd.DataFrame([{
    'age': 58, 'job': 'retired', 'marital': 'divorced',
    'education': 'tertiary', 'default': 'no', 'balance': 5000,
    'housing': 'no', 'loan': 'no', 'contact': 'cellular',
    'day': 20, 'month': 'aug', 'duration': 500, 'campaign': 2,
    'pdays': -1, 'previous': 0, 'poutcome': 'unknown'
}])

for i, cliente in enumerate([cliente1, cliente2], 1):
    cliente['contacted_before'] = (cliente['pdays'] != -1).astype(int)
    cliente['pdays_clean']      = cliente['pdays'].clip(lower=0)
    cliente = cliente.drop(columns=['pdays'])
    pred = le.inverse_transform(modelo.predict(cliente))[0]
    print(f'Cliente {i}: {pred}')

Cliente 1: no
Cliente 2: yes
